# EngGPT Guidelines
**Author:** EngGPT-Team

**Date:**  20/01/2026 

In [ ]:
%pip install boto3 python-dotenv

In [ ]:
import boto3
import os
import json

# AWS EngGPT2 call

Environment variables are loaded from a `.env` file located in the same folder as this notebook. Please ensure the file includes:
```text
AWS_ACCESS_KEY_ID=<access-key-id>
AWS_SECRET_ACCESS_KEY=<secret-access-key>
AWS_SESSION_TOKEN=<session-token>
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
AWS_ACCESS_KEY=os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY=os.getenv("AWS_SECRET_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
ENDPOINT_NAME = 'g6e-test'
REGION_NAME='eu-central-1' 

In [ ]:
runtime = boto3.client(
    'sagemaker-runtime',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name='eu-central-1' 
)

## Use chat completions
The Chat Completions API is designed to mimic a conversation, making it more intuitive for controlling the model's behavior and context.
You send an array of message objects, each with a role and content.
| Role | Purpose | Example |
| ----------- | ----------- | ----------- |
| system | "Sets the global context, personality, and instructions for the assistant." | "You are a helpful pirate who answers all questions in nautical terms." |
| user | The latest query or request from the end-user. | "What are the three rules of thermodynamics?" |
| assistant | The model's previous response (used to maintain conversation history). | "Ahoy! They be the first, second, and third laws of the sea!" |

### With reasoning

In [ ]:
messages = [{"role":"system", "content":"Give a short but exhaustive answer."}, 
            {"role":"user", "content":"Who are you"} ]
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages})
)
response_json = json.loads(response['Body'].read().decode())
print(f"##REASONING##\n{response_json['choices'][0]['message']['reasoning']}")
print(f"##RESPONSE##\n{response_json['choices'][0]['message']['content']}")

### Reasoning Ita

In [ ]:

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages,
                     "chat_template_kwargs":{"reasoning_lang":"ita"}})
)
response_json = json.loads(response['Body'].read().decode())

print(f"##REASONING##\n{response_json['choices'][0]['message']['reasoning']}")
print(f"##RESPONSE##\n{response_json['choices'][0]['message']['content']}")

### Reasoning Turbo

In [ ]:

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages,
                     "chat_template_kwargs":{"enable_turbo": True}})
)
response_json = json.loads(response['Body'].read().decode())

print(f"##REASONING##\n{response_json['choices'][0]['message']['reasoning']}")
print(f"##RESPONSE##\n{response_json['choices'][0]['message']['content']}")

### Without reasoning

In [ ]:

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages,
                     "chat_template_kwargs":{"enable_thinking":False}})
)
response_json = json.loads(response['Body'].read().decode())

print(f"##RESPONSE##\n{response_json['choices'][0]['message']['content']}")

### Streaming Tokens

In [ ]:
messages = [{"role":"system", "content":"Sei EngGPT, l'assistente virtuale sviluppato da Engineering Ingegneria Informatica."}, 
            {"role":"user", "content":"Ciao, chi sei?"} ]

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({
        "messages": messages,
        "stream": True,
        "chat_template_kwargs": {"reasoning_lang": "ita"}
    })
)

buffer = ""

for event in response['Body']:
    chunk_raw = event.decode()
    buffer += chunk_raw  # Accumula nel buffer
    
    # Processa solo le righe complete (terminate da \n)
    while "\n" in buffer:
        line, buffer = buffer.split("\n", 1)
        line = line.strip()
        
        if not line.startswith("data: "):
            continue
            
        content_str = line[6:].strip()
        
        if content_str == "[DONE]":
            break
            
        try:
            line_json = json.loads(content_str)
            delta = line_json['choices'][0].get('delta', {})

            if 'reasoning' in delta:
                print(delta['reasoning'], end="", flush=True)
            
            if 'content' in delta:
                print(delta['content'], end="", flush=True)
                
        except json.JSONDecodeError:
            continue

### Reasoning with thinking budget
**Thinking budget**: A limit on how many tokens the LLM can use for thinking. 

In [ ]:
budget = 100
messages = [{"role":"system", "content":"Rispondi in modo breve ma esaustivo"}, 
            {"role":"user", "content":"Chi sei?"} ]
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages, 
                     "stop":"</think>", 
                     "max_tokens":budget})
)
response_json = json.loads(response['Body'].read().decode())
reasoning_resp = response_json["choices"][0]["message"]["content"]
print(f"##REASONING##\n{reasoning_resp}")

messages.append({"role":"assistant", 
                 "content":f"{reasoning_resp}</think>"})
response= runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"messages": messages})
)
response_json = json.loads(response['Body'].read().decode())
print(f"##RESPONSE##\n{response_json['choices'][0]['message']['content']}")

## Use completions
The Completions API is simpler, taking a single string of text as input. It's now primarily used for older, non-chat-optimized models or for very simple, single-prompt tasks.
The entire context, instructions, and query must be contained in the prompt string.

Care must be taken with the individual models' special tokens; these chat templates are valid for the current EngGPT2 models.

In [ ]:
messages = [{"role":"system", "content":"Rispondi in modo breve ma esaustivo"}, 
            {"role":"user", "content":"Chi sei?"} ]
raw_prompt = ""
for message in messages: 
    raw_prompt+=f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"

# Reasoning injection
reasoning_resp = "Okay, the user is asking: Who are you? I have to respond clearly and concisely. First of all, I should mention my name, Enggpt, and that I am an assistant to developed by Engineering S.p.A. It is important to highlight that I am an I -intelligent."
raw_prompt += f"""<|im_start|>assistant\n/reasoning_en\n<think>\n{reasoning_resp}\n</think>\n\n"""

response= runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps({"prompt": raw_prompt,
                     "max_tokens":1000})
)
response_json = json.loads(response['Body'].read().decode())

print(f"##RESPONSE##\n{response_json['choices'][0]['text']}")

